In [13]:
# Import necessary libraries for data manipulation and LightGBM
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

In [14]:
# Load the F1 Strategy Dataset
df = pd.read_csv('f1_strategy_dataset_v4.csv')

# Display the first few rows
df.head()

,Driver,LapNumber,Compound,Stint,TyreLife,Position,LapTime (s),Race,Year,LapTime_Delta,Cumulative_Degradation,PitStop,PitNextLap,RaceProgress,Normalized_TyreLife,Position_Change
0,ALB,1,MEDIUM,1,2.0,17,100.625,Abu Dhabi Grand Prix,2023,0.000,0.000,0,0,0.017241,0.117647,0.0
1,ALB,2,MEDIUM,1,3.0,18,93.560,Abu Dhabi Grand Prix,2023,-7.065,-7.065,0,0,0.034483,0.176471,-1.0
2,ALB,3,MEDIUM,1,4.0,18,91.768,Abu Dhabi Grand Prix,2023,-1.792,-8.857,0,0,0.051724,0.235294,0.0
3,ALB,4,MEDIUM,1,5.0,18,91.591,Abu Dhabi Grand Prix,2023,-0.177,-9.034,0,0,0.068966,0.294118,0.0
4,ALB,5,MEDIUM,1,6.0,18,91.422,Abu Dhabi Grand Prix,2023,-0.169,-9.203,0,0,0.086207,0.352941,0.0


In [15]:
# Basic statistics
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 101371 entries, 0 to 101370
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Driver                  101371 non-null  str    
 1   LapNumber               101371 non-null  int64  
 2   Compound                101305 non-null  str    
 3   Stint                   101371 non-null  int64  
 4   TyreLife                101371 non-null  float64
 5   Position                101371 non-null  int64  
 6   LapTime (s)             101371 non-null  float64
 7   Race                    101371 non-null  str    
 8   Year                    101371 non-null  int64  
 9   LapTime_Delta           101371 non-null  float64
 10  Cumulative_Degradation  101371 non-null  float64
 11  PitStop                 101371 non-null  int64  
 12  PitNextLap              101371 non-null  int64  
 13  RaceProgress            101371 non-null  float64
 14  Normalized_TyreLife     101371 

In [16]:
df.describe()

,LapNumber,Stint,TyreLife,Position,LapTime (s),Year,LapTime_Delta,Cumulative_Degradation,PitStop,PitNextLap,RaceProgress,Normalized_TyreLife,Position_Change
count,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000
mean,30.444841,2.046394,14.549339,9.759132,92.587188,2023.589685,-0.203891,-29.550051,0.251581,0.254797,0.432618,0.386521,-0.004636
std,18.146942,0.948797,10.313385,5.406456,33.231414,1.098518,45.344910,70.235759,0.433924,0.435749,0.258129,0.259906,3.912725
min,1.000000,1.000000,1.000000,1.000000,67.012000,2022.000000,-2403.895000,-274.564000,0.000000,0.000000,0.012821,0.012821,-18.000000
25%,15.000000,1.000000,7.000000,5.000000,82.021000,2023.000000,-7.253000,-51.054500,0.000000,0.000000,0.210526,0.172414,-2.000000
50%,30.000000,2.000000,13.000000,10.000000,91.167000,2024.000000,-0.027000,-21.678000,0.000000,0.000000,0.421053,0.333333,0.000000
75%,45.000000,3.000000,20.000000,14.000000,99.356000,2025.000000,5.705000,-3.725500,1.000000,1.000000,0.631579,0.562500,2.000000
max,78.000000,8.000000,78.000000,20.000000,2526.253000,2025.000000,2433.472000,2412.431000,1.000000,1.000000,1.000000,1.000000,18.000000


In [17]:

# Target distribution
df['PitNextLap'].value_counts()
df['PitNextLap'].value_counts(normalize=True)



PitNextLap
0    0.745203
1    0.254797
Name: proportion, dtype: float64

In [18]:
# Missing values
df.isnull().sum()

Driver                     0
LapNumber                  0
Compound                  66
Stint                      0
TyreLife                   0
Position                   0
LapTime (s)                0
Race                       0
Year                       0
LapTime_Delta              0
Cumulative_Degradation     0
PitStop                    0
PitNextLap                 0
RaceProgress               0
Normalized_TyreLife        0
Position_Change            0
dtype: int64

In [19]:
# Time-series aware train/test split
# Each unique combination of Driver + Race + Year forms a sequence ordered by LapNumber
# We split each sequence at 70/30 to preserve temporal order

# Group by driver-race-year to identify sequences
sequences = df.groupby(['Driver', 'Race', 'Year'])

train_sequences = []
test_sequences = []

for name, group in sequences:
    # Split each sequence at 70/30
    split_idx = int(len(group) * 0.7)
    train_sequences.append(group.iloc[:split_idx])
    test_sequences.append(group.iloc[split_idx:])

# Concatenate all train and test sequences
train_df = pd.concat(train_sequences, ignore_index=True)
test_df = pd.concat(test_sequences, ignore_index=True)

print(f"Total rows: {len(df)}")
print(f"Train size: {len(train_df)} ({len(train_df)/len(df):.2%})")
print(f"Test size: {len(test_df)} ({len(test_df)/len(df):.2%})")
print(f"\nTrain/Test ratio: {len(train_df)/len(test_df):.2f}")

# Verify temporal order is preserved
print(f"\nUnique sequences in train: {train_df.groupby(['Driver', 'Race', 'Year']).ngroups}")
print(f"Unique sequences in test: {test_df.groupby(['Driver', 'Race', 'Year']).ngroups}")

Total rows: 101371
Train size: 70187 (69.24%)
Test size: 31184 (30.76%)

Train/Test ratio: 2.25

Unique sequences in train: 1838
Unique sequences in test: 1848


In [20]:
train_df = train_df.drop(['Driver', 'Race'], axis=1)

In [21]:
# Convert Compound column to categorical with specified order
categories = ['SOFT', 'MEDIUM', 'HARD']
dtype = pd.CategoricalDtype(categories=categories, ordered=True)
train_df['Compound'] = train_df['Compound'].astype(dtype)

# Verify the conversion
train_df['Compound'].cat.categories.tolist()
train_df['Compound'].cat.codes.value_counts()

 1    29303
 2    29116
 0     7268
-1     4500
Name: count, dtype: int64

In [ ]:
# Impute missing values


In [22]:
# Encode Compound column with OrdinalEncoder
encoder = OrdinalEncoder()
train_df['Compound_encoded'] = encoder.fit_transform(train_df['Compound'].values.reshape(-1, 1)).flatten()

# Drop the original Compound column
train_df.drop('Compound', axis=1, inplace=True)

In [25]:
train_df

,LapNumber,Stint,TyreLife,Position,LapTime (s),Year,LapTime_Delta,Cumulative_Degradation,PitStop,PitNextLap,RaceProgress,Normalized_TyreLife,Position_Change,Compound_encoded
0,1,1,2.0,17,100.625,2023,0.000,0.000,0,0,0.017241,0.117647,0.0,1.0
1,2,1,3.0,18,93.560,2023,-7.065,-7.065,0,0,0.034483,0.176471,-1.0,1.0
2,3,1,4.0,18,91.768,2023,-1.792,-8.857,0,0,0.051724,0.235294,0.0,1.0
3,4,1,5.0,18,91.591,2023,-0.177,-9.034,0,0,0.068966,0.294118,0.0,1.0
4,5,1,6.0,18,91.422,2023,-0.169,-9.203,0,0,0.086207,0.352941,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70182,34,2,21.0,18,101.387,2024,15.900,-10.744,0,0,0.447368,0.304348,-2.0,1.0
70183,35,2,22.0,19,101.783,2024,-17.831,-10.348,0,0,0.460526,0.318841,-1.0,1.0
70184,36,3,1.0,19,126.093,2024,24.810,52.277,1,0,0.473684,0.030303,-4.0,0.0
70185,37,3,2.0,19,100.161,2024,-3.767,26.345,0,0,0.486842,0.060606,0.0,0.0
